In [ ]:
# %pip install opencv-python scikit-image

In [1]:
import numpy as np
import os
import cv2
from skimage.feature import hog

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [2]:
DATASET_PATH = "../data/hiragana"
# FIX 1: cv2.resize expects (width, height), so a consistent (64, 64) square
# avoids the asymmetric (84, 83) which was likely a typo, and ensures
# HOG cell/block math works out cleanly.
IMG_SIZE = (64, 64)

In [3]:
def extract_hog(img):
    # FIX 2: Added channel_axis=None to explicitly tell skimage this is a
    # grayscale 2D array (required in newer skimage versions).
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        channel_axis=None
    )
    return features

In [4]:
data = []
labels = []
label_names = []

for i, label in enumerate(sorted(os.listdir(DATASET_PATH))):
    folder_path = os.path.join(DATASET_PATH, label)
    
    if not os.path.isdir(folder_path):
        continue
    
    label_names.append(label)
    
    for file in os.listdir(folder_path):
        if not file.lower().endswith(".jpg"):  # FIX 3: .lower() handles .JPG/.JPEG variants
            continue
        
        img_path = os.path.join(folder_path, file)
        
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # FIX 4: Guard against failed reads (corrupted/missing files)
        if img is None:
            print(f"Warning: could not read {img_path}, skipping.")
            continue
        
        img = cv2.resize(img, IMG_SIZE)
        
        features = extract_hog(img)
        
        data.append(features)
        labels.append(i)

X = np.array(data)
y = np.array(labels)

print("Dataset shape:", X.shape)
print("Number of classes:", len(label_names))

Dataset shape: (4600, 1764)
Number of classes: 46


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

In [7]:
param_grid = {
    "svm__C": [1, 10, 50],
    "svm__gamma": ["scale", 0.01, 0.001],
    "svm__kernel": ["rbf"]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Parameters: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}


In [8]:
y_pred = grid.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_names))

Accuracy: 0.9782608695652174
              precision    recall  f1-score   support

          aa       1.00      1.00      1.00        20
         chi       1.00      1.00      1.00        20
          ee       0.95      0.90      0.92        20
          fu       1.00      0.95      0.97        20
          ha       1.00      0.95      0.97        20
          he       1.00      1.00      1.00        20
          hi       1.00      1.00      1.00        20
          ho       1.00      1.00      1.00        20
          ii       1.00      1.00      1.00        20
          ka       1.00      0.95      0.97        20
          ke       1.00      0.90      0.95        20
          ki       1.00      0.95      0.97        20
          ko       1.00      1.00      1.00        20
          ku       1.00      0.90      0.95        20
          ma       1.00      1.00      1.00        20
          me       1.00      1.00      1.00        20
          mi       1.00      1.00      1.00        2

In [11]:
def predict_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    
    # FIX 5: Guard against failed image read — this was the direct cause
    # of the OpenCV assertion error (!ssize.empty()). cv2.imread returns
    # None when the file doesn't exist or can't be decoded, and passing
    # None into cv2.resize raises the cryptic assertion error.
    if img is None:
        print(f"Error: could not load image at '{path}'. Check the path and file format.")
        return
    
    img = cv2.resize(img, IMG_SIZE)
    
    features = extract_hog(img).reshape(1, -1)
    
    pred = grid.predict(features)[0]
    
    print("Prediction:", label_names[pred])

# Example — replace with a real path to a hiragana image
predict_image("../data/hiragana/chi/drawing_20250805_230844.jpg")

Prediction: chi
